# Notebook 04: Hybrid Pipeline, Gaussian Mixture Attention with Convolutional Classification

**Course:** DS 4889 Machine Learning, United International University

**Course outcomes:** CO4 (feature construction and transformation), CO5 (sound validation and interpretation), CO7 (unsupervised pattern discovery), CO8 (comparison of alternative approaches)

**Lecture mapping:** Lecture 06 (Gaussian Mixture Models and the Expectation Maximisation algorithm), Lectures 08, 09 and 11 (neural networks and convolutional architectures)

## Hypothesis

The hypothesis under test is stated explicitly so that the experiment can falsify it.

*Supplying the convolutional network with a spatial prior derived from an unsupervised intensity model
reduces the burden of learning where to look. Because the network no longer has to discover the
location of the abnormality from the label signal alone, more of its capacity is available for
discriminating between lesion types. This should improve sample efficiency, that is reach a given
validation accuracy in fewer epochs, and should reduce confusion between classes that differ mainly
in the location of the lesion rather than in its texture, most notably the glioma and meningioma
pair.*

The prior is obtained by fitting a Gaussian Mixture Model to the intensity histogram of each slice.
The mixture treats the histogram as a weighted sum of Gaussian densities whose parameters are
estimated by Expectation Maximisation. Because brain tissue classes occupy broadly separable intensity
bands, the fitted components approximate background, normal parenchyma and hyperintense tissue. The
hyperintense component is selected as the lesion candidate, cleaned morphologically, blurred into a
soft attention map and used to attenuate the image outside the candidate region. Attenuation rather
than hard masking is used because anatomical context is itself diagnostic: removing everything outside
the mask would discard the information needed to tell a meningioma at the brain margin from a
pituitary lesion at the skull base.

The segmentation stage receives no labels at any point, so it introduces no information leakage from
the targets into the inputs.

## Controlled comparison design

The experiment is constructed so that exactly one factor differs from notebook 03.

| Factor | Notebook 03 | Notebook 04 |
| --- | --- | --- |
| Architecture | `cnn_model.build_cnn` | `cnn_model.build_cnn`, identical |
| Parameter count | reference | asserted equal to the reference |
| Random seed | `config.RANDOM_SEED` | `config.RANDOM_SEED` |
| Train and validation split | stratified, same seed | stratified, same seed |
| Preprocessing | resize, CLAHE, normalise | resize, CLAHE, normalise |
| Augmentation | same policy | same policy |
| Optimiser, schedule, callbacks | same | same |
| **Input transformation** | **none** | **GMM attention weighting** |

Holding all other factors fixed is what allows any difference in test performance to be attributed to
the segmentation stage, and what makes the paired McNemar test in notebook 05 a valid instrument.


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

sys.path.append('..')
from src import (
    config,
    data_loader,
    preprocessing,
    gmm_segmentation,
    cnn_model,
    evaluate,
    explainability,
)

# The same seed call as notebook 03. Reproducing the initialisation and the shuffling order is
# a precondition for attributing any performance difference to the input transformation.
cnn_model.set_global_seed()

sns.set_theme(style='whitegrid', context='notebook')

print('TensorFlow version:', tf.__version__)

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print('GPU devices visible:', [device.name for device in gpu_devices])
else:
    print('No GPU is visible; training will proceed on the CPU and will take longer.')

print('Random seed:', config.RANDOM_SEED)
print('Mixture components per image:', config.GMM_N_COMPONENTS)
print('Background retention weight:', config.ROI_BACKGROUND_WEIGHT)


## Reproducing the data splits

The partition of the training folder into training and validation subsets is produced by
`data_loader.stratified_validation_split`, which passes `config.RANDOM_SEED` to scikit-learn as the
`random_state` argument. Given the same input array, the same split fraction and the same seed, the
function is deterministic and returns exactly the same partition as in notebook 03. Images are also
read in a sorted and therefore stable order by `data_loader.load_split`, so the arrays entering the
split are identical as well.

This matters for two reasons. First, if the two models were trained on different partitions, a
difference in test performance could reflect a lucky or unlucky validation split rather than the
segmentation stage. Second, notebook 05 applies McNemar's test, which requires that both models be
evaluated on precisely the same test samples in precisely the same order; that requirement is met only
if the loading and splitting procedure is reproduced exactly.


In [ ]:
# Identical to notebook 03. Keep LIMIT_PER_CLASS at the same value used there, otherwise the two
# experiments are no longer comparable.
LIMIT_PER_CLASS = None

load_start = time.time()

x_train_raw, y_train_full, train_paths = data_loader.load_split(
    config.TRAIN_DIR,
    image_size=config.IMAGE_SIZE,
    grayscale=False,
    limit_per_class=LIMIT_PER_CLASS,
    show_progress=True,
)

x_test_raw, y_test, test_paths = data_loader.load_split(
    config.TEST_DIR,
    image_size=config.IMAGE_SIZE,
    grayscale=False,
    limit_per_class=LIMIT_PER_CLASS,
    show_progress=True,
)

x_train_full = preprocessing.preprocess_batch(x_train_raw, image_size=config.IMAGE_SIZE)
x_test = preprocessing.preprocess_batch(x_test_raw, image_size=config.IMAGE_SIZE)

x_train, x_val, y_train, y_val = data_loader.stratified_validation_split(x_train_full, y_train_full)

y_train_cat = to_categorical(y_train, num_classes=config.NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=config.NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=config.NUM_CLASSES)

class_weights = data_loader.compute_class_weights(y_train)

print('Loading and preprocessing completed in {0:.1f} seconds.'.format(time.time() - load_start))
print('Training subset:  ', x_train.shape)
print('Validation subset:', x_val.shape)
print('Test set:         ', x_test.shape)
print('Class weights:', {config.CLASS_LABELS[k]: round(v, 4) for k, v in class_weights.items()})


## Applying the segmentation stage

One Gaussian Mixture Model is fitted per image. The model is fitted on the intensity values of that
slice alone, which is deliberate: scanner gain, sequence weighting and patient anatomy vary between
acquisitions, so a mixture fitted on the pooled histogram of the whole dataset would describe no
individual scan well. Fitting per image makes the component boundaries adapt to the intensity scale of
the slice at hand.

The price of that choice is computational. Expectation Maximisation is run to convergence on roughly
fifty thousand pixel intensities for every image, and the transformation is applied to every training,
validation and test sample. This is by a wide margin the dominant cost of the hybrid pipeline, far
exceeding the cost of a training epoch, and it is incurred again at inference time for every new scan.

Because the transformation is deterministic given the seed, recomputing it on each run would waste
time without changing the result. The transformed arrays are therefore written once to a compressed
archive in `config.CACHE_DIR` and reloaded on subsequent runs. The cache is only accepted when the
stored array shapes match the shapes expected from the current split, which protects against silently
training on arrays produced under a different value of `LIMIT_PER_CLASS` or a different image size.


In [ ]:
cache_path = config.CACHE_DIR / 'hybrid_arrays.npz'
expected_shapes = {
    'x_train': tuple(x_train.shape),
    'x_val': tuple(x_val.shape),
    'x_test': tuple(x_test.shape),
}

x_train_hybrid = None
x_val_hybrid = None
x_test_hybrid = None
transform_seconds = None

if cache_path.exists():
    with np.load(cache_path) as cached:
        has_all_keys = all(key in cached.files for key in expected_shapes)
        shapes_match = has_all_keys and all(
            tuple(cached[key].shape) == expected_shapes[key] for key in expected_shapes
        )
        if shapes_match:
            x_train_hybrid = cached['x_train']
            x_val_hybrid = cached['x_val']
            x_test_hybrid = cached['x_test']
            print('Loaded cached hybrid arrays from', cache_path)
        else:
            print('Cache found but the stored shapes do not match the current split; recomputing.')
else:
    print('No cache found at', cache_path)

if x_train_hybrid is None:
    transform_start = time.time()

    x_train_hybrid = gmm_segmentation.transform_batch_for_hybrid(
        x_train, show_progress=True, description='GMM transform, training split'
    )
    x_val_hybrid = gmm_segmentation.transform_batch_for_hybrid(
        x_val, show_progress=True, description='GMM transform, validation split'
    )
    x_test_hybrid = gmm_segmentation.transform_batch_for_hybrid(
        x_test, show_progress=True, description='GMM transform, test split'
    )

    transform_seconds = time.time() - transform_start

    np.savez_compressed(
        cache_path,
        x_train=x_train_hybrid,
        x_val=x_val_hybrid,
        x_test=x_test_hybrid,
    )

    total_images = len(x_train_hybrid) + len(x_val_hybrid) + len(x_test_hybrid)
    print('Transformed {0} images in {1:.1f} seconds ({2:.1f} minutes).'.format(
        total_images, transform_seconds, transform_seconds / 60.0
    ))
    print('Mean cost per image: {0:.4f} seconds.'.format(transform_seconds / total_images))
    print('Cache written to', cache_path)

print('Hybrid training tensor:  ', x_train_hybrid.shape, x_train_hybrid.dtype)
print('Hybrid validation tensor:', x_val_hybrid.shape)
print('Hybrid test tensor:      ', x_test_hybrid.shape)


In [ ]:
# An unsupervised stage can fail silently: it will always return a mask of some kind, and a
# degenerate mask that selects the skull or the background would degrade rather than assist the
# classifier. One example per class is therefore inspected before any training is performed.
figure, axes = plt.subplots(config.NUM_CLASSES, 3, figsize=(11, 14.5))

for row, class_index in enumerate(range(config.NUM_CLASSES)):
    position = int(np.flatnonzero(y_train == class_index)[0])
    original = x_train[position]
    segmentation = gmm_segmentation.segment_image(original)
    weighted = x_train_hybrid[position]

    axes[row, 0].imshow(np.clip(original, 0.0, 1.0))
    axes[row, 0].set_title('{0}: preprocessed'.format(config.CLASS_LABELS[class_index]), fontsize=11)

    axes[row, 1].imshow(segmentation['mask'], cmap='gray')
    axes[row, 1].set_title('GMM mask, area {0:.3f}'.format(segmentation['area_ratio']), fontsize=11)

    axes[row, 2].imshow(np.clip(weighted, 0.0, 1.0))
    axes[row, 2].set_title('Attention weighted input', fontsize=11)

    for column in range(3):
        axes[row, column].axis('off')

figure.suptitle('Hybrid transformation, one representative case per class', fontsize=14)
figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'hybrid_transformation_examples.png')
plt.show()


## Model definition

The network is constructed by the same `cnn_model.build_cnn` call used in notebook 03, with the same
default arguments. The architecture is therefore identical layer for layer, filter for filter and
parameter for parameter; only the name of the model differs, which has no effect on computation. The
input shape is unchanged because the attention weighting rescales intensities and does not alter the
tensor geometry.

The equality of the parameter counts is asserted programmatically rather than assumed. If the
architectures were to diverge, for example through an accidental edit to the builder between the two
runs, the comparison would be confounded by capacity and the conclusion of the study would be invalid.
Making the check an explicit assertion converts a silent methodological error into a visible failure.


In [ ]:
hybrid_model = cnn_model.build_cnn(name='gmm_cnn_hybrid')
hybrid_model = cnn_model.compile_model(hybrid_model, learning_rate=config.LEARNING_RATE)

hybrid_parameter_count = int(hybrid_model.count_params())
print('Hybrid model parameters: {0:,}'.format(hybrid_parameter_count))
print('Hybrid model layers:', len(hybrid_model.layers))

baseline_path = config.MODELS_DIR / config.BASELINE_MODEL_NAME
try:
    reference_model = tf.keras.models.load_model(baseline_path)
    reference_parameter_count = int(reference_model.count_params())
    print('Baseline model parameters: {0:,}'.format(reference_parameter_count))
    assert hybrid_parameter_count == reference_parameter_count, (
        'The two architectures differ in capacity, so the comparison would be confounded.'
    )
    print('Parameter counts are equal; the comparison is controlled for capacity.')
    del reference_model
except Exception as error:
    print('The baseline model could not be loaded for comparison:', type(error).__name__, error)
    print('Run notebook 03 first if a direct parameter check against the baseline is required.')


In [ ]:
train_generator = preprocessing.make_training_generator(
    x_train_hybrid, y_train_cat, batch_size=config.BATCH_SIZE
)

EPOCHS = config.EPOCHS
steps_per_epoch = max(1, len(x_train_hybrid) // config.BATCH_SIZE)
checkpoint_path = config.MODELS_DIR / config.HYBRID_MODEL_NAME

callbacks = cnn_model.build_callbacks(checkpoint_path, monitor='val_accuracy')

print('Maximum epochs:  ', EPOCHS)
print('Steps per epoch: ', steps_per_epoch)
print('Checkpoint path: ', checkpoint_path)

training_start = time.time()

history = hybrid_model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=(x_val_hybrid, y_val_cat),
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)

hybrid_training_seconds = time.time() - training_start

print()
print('Training wall clock time: {0:.1f} seconds ({1:.1f} minutes).'.format(
    hybrid_training_seconds, hybrid_training_seconds / 60.0
))


In [ ]:
evaluate.plot_training_history(
    history,
    title_prefix='Hybrid GMM and CNN ',
    save_path=config.FIGURES_DIR / 'hybrid_training_history.png',
)
plt.show()

epochs_completed = len(history.history['loss'])
best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
best_val_accuracy = float(np.max(history.history['val_accuracy']))
final_train_accuracy = float(history.history['accuracy'][-1])

print('Epochs completed before stopping: {0} of a maximum of {1}'.format(epochs_completed, EPOCHS))
print('Best validation accuracy: {0:.4f} at epoch {1}'.format(best_val_accuracy, best_epoch))
print('Final training accuracy:  {0:.4f}'.format(final_train_accuracy))

# The epoch at which the best validation accuracy occurs is the sample efficiency signal named in
# the hypothesis: reaching the same quality sooner indicates an easier optimisation problem.
try:
    baseline_history = evaluate.load_metrics('baseline_history.json')
    print()
    print('Baseline best epoch: {0}, best validation accuracy {1:.4f}'.format(
        baseline_history['best_epoch'], baseline_history['best_val_accuracy']
    ))
    print('Hybrid best epoch:   {0}, best validation accuracy {1:.4f}'.format(
        best_epoch, best_val_accuracy
    ))
except FileNotFoundError:
    print('No stored baseline history was found; run notebook 03 to enable this comparison.')


## Evaluation

The hybrid model is now evaluated on the same held out test split, in the same order, with the same
metric functions used in notebook 03. Reusing the shared implementations in `src.evaluate` rather than
recomputing the quantities locally removes the possibility that a difference between the two sets of
figures originates in a difference of definition, for example a change of averaging mode, rather than
in a difference of model behaviour.


In [ ]:
hybrid_probabilities = hybrid_model.predict(x_test_hybrid, batch_size=config.BATCH_SIZE, verbose=0)
hybrid_predictions = np.argmax(hybrid_probabilities, axis=1)

hybrid_metrics = evaluate.compute_metrics(y_test, hybrid_predictions, 'GMM and CNN hybrid')

metrics_series = pd.Series(
    {key: value for key, value in hybrid_metrics.items() if key != 'model'},
    name='GMM and CNN hybrid',
)
print('Headline metrics on the held out test set')
display(metrics_series.to_frame().round(4))

hybrid_per_class = evaluate.per_class_metrics(y_test, hybrid_predictions)
print('Per class metrics')
display(hybrid_per_class)

figure, axes = plt.subplots(1, 2, figsize=(15, 5.8))

evaluate.plot_confusion_matrix(
    y_test,
    hybrid_predictions,
    class_labels=config.CLASS_LABELS,
    title='Hybrid GMM and CNN: counts',
    normalise=False,
    ax=axes[0],
)

evaluate.plot_confusion_matrix(
    y_test,
    hybrid_predictions,
    class_labels=config.CLASS_LABELS,
    title='Hybrid GMM and CNN: row normalised',
    normalise=True,
    ax=axes[1],
)

figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'hybrid_confusion_matrices.png')
plt.show()

figure, ax = plt.subplots(figsize=(7, 6))
ax, hybrid_auc_scores = evaluate.plot_roc_curves(
    y_test,
    hybrid_probabilities,
    class_labels=config.CLASS_LABELS,
    title='Hybrid GMM and CNN: one versus rest ROC',
    ax=ax,
)
figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'hybrid_roc_curves.png')
plt.show()

print('Area under the curve by class')
for label, score in hybrid_auc_scores.items():
    print('  {0:<16} {1:.4f}'.format(label, score))


In [ ]:
hybrid_clinical = evaluate.clinical_error_summary(y_test, hybrid_predictions, healthy_index=2)

print('Clinically weighted error summary')
for key, value in hybrid_clinical.items():
    print('  {0:<24} {1}'.format(key, value))

print()
hybrid_accuracy_interval = evaluate.bootstrap_confidence_interval(
    y_test, hybrid_predictions, metric='accuracy', n_iterations=2000, confidence=0.95
)
hybrid_f1_interval = evaluate.bootstrap_confidence_interval(
    y_test, hybrid_predictions, metric='f1_weighted', n_iterations=2000, confidence=0.95
)

for interval in (hybrid_accuracy_interval, hybrid_f1_interval):
    print('{0}: {1:.4f} with a {2:.0%} interval of [{3:.4f}, {4:.4f}]'.format(
        interval['metric'],
        interval['point_estimate'],
        interval['confidence'],
        interval['lower_bound'],
        interval['upper_bound'],
    ))

# An overlapping pair of intervals is not by itself proof of equivalence, but a non overlapping
# pair is a strong indication that the difference is not attributable to sampling alone. The
# formal paired test is performed in notebook 05.
try:
    baseline_metrics = evaluate.load_metrics('baseline_metrics.json')
    print()
    print('Baseline accuracy {0:.4f} against hybrid accuracy {1:.4f}.'.format(
        baseline_metrics['accuracy'], hybrid_metrics['accuracy']
    ))
    print('Baseline weighted F1 {0:.4f} against hybrid weighted F1 {1:.4f}.'.format(
        baseline_metrics['f1_weighted'], hybrid_metrics['f1_weighted']
    ))
except FileNotFoundError:
    print('No stored baseline metrics were found; run notebook 03 to enable this comparison.')


## Does the network attend to the segmented region?

A gain in accuracy would be consistent with the hypothesis but would not confirm the mechanism that
the hypothesis proposes. The network could improve for an unrelated reason, for example because the
attenuation acts as a generic regulariser similar to random erasing, while ignoring the region the
mixture model identified. Conversely, an absence of improvement could mean either that the prior is
unhelpful or that the network simply never uses it.

Gradient weighted Class Activation Mapping provides direct evidence on this point. The method weights
the feature maps of the deepest convolutional layer by the gradient of the predicted class score,
producing a coarse map of the regions that drove the decision. Comparing that map with the Gaussian
Mixture mask through the intersection over union gives a quantitative measure of whether the evidence
the network used coincides with the region the unsupervised stage selected.

A high overlap supports the designed mechanism: the attention map biased the network towards the
lesion and the network made its decision there. A low overlap falsifies the mechanism even if accuracy
improved, and indicates that any gain must be explained in another way. Interpreting the number
requires one caveat: the Grad-CAM map has the spatial resolution of the final convolutional feature
map, which is coarse relative to a small lesion, so the intersection over union is bounded well below
one even under perfect agreement. The value is meaningful as a relative and comparative quantity
rather than as an absolute segmentation score.


In [ ]:
rng = np.random.default_rng(config.RANDOM_SEED)
display_indices = rng.choice(len(x_test_hybrid), size=6, replace=False)

figure, axes = plt.subplots(6, 3, figsize=(11, 21))

for row, index in enumerate(display_indices):
    explanation = explainability.generate_explanation(hybrid_model, x_test_hybrid[index])
    segmentation = gmm_segmentation.segment_image(x_test[index])
    iou = explainability.heatmap_mask_agreement(
        explanation['heatmap'], segmentation['mask'], threshold=0.5
    )

    axes[row, 0].imshow(np.clip(x_test[index], 0.0, 1.0))
    axes[row, 0].set_title(
        'True: {0}'.format(config.CLASS_LABELS[int(y_test[index])]), fontsize=11
    )

    axes[row, 1].imshow(explanation['overlay'])
    axes[row, 1].set_title(
        'Grad-CAM: {0} ({1:.2f}), IoU {2:.3f}'.format(
            explanation['class_label'], explanation['confidence'], iou
        ),
        fontsize=11,
    )

    axes[row, 2].imshow(segmentation['mask'], cmap='gray')
    axes[row, 2].set_title('GMM mask, IoU {0:.3f}'.format(iou), fontsize=11)

    for column in range(3):
        axes[row, column].axis('off')

figure.suptitle('Agreement between network attention and the unsupervised mask', fontsize=14)
figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'hybrid_gradcam_mask_agreement.png')
plt.show()

# Six cases are an illustration rather than evidence. The mean over a larger sample is the
# quantity that supports or undermines the mechanism.
AGREEMENT_SAMPLE_SIZE = min(100, len(x_test_hybrid))
agreement_indices = rng.choice(len(x_test_hybrid), size=AGREEMENT_SAMPLE_SIZE, replace=False)

agreement_scores = []
agreement_start = time.time()
for index in agreement_indices:
    heatmap, _, _ = explainability.compute_gradcam_heatmap(hybrid_model, x_test_hybrid[index])
    mask = gmm_segmentation.segment_image(x_test[index])['mask']
    agreement_scores.append(explainability.heatmap_mask_agreement(heatmap, mask, threshold=0.5))

agreement_scores = np.asarray(agreement_scores, dtype=np.float64)

print('Sample size: {0} test images, computed in {1:.1f} seconds.'.format(
    AGREEMENT_SAMPLE_SIZE, time.time() - agreement_start
))
print('Mean intersection over union:   {0:.4f}'.format(float(agreement_scores.mean())))
print('Median intersection over union: {0:.4f}'.format(float(np.median(agreement_scores))))
print('Standard deviation:             {0:.4f}'.format(float(agreement_scores.std())))
print('Proportion of cases above 0.10: {0:.3f}'.format(float((agreement_scores > 0.10).mean())))


In [ ]:
hybrid_metrics_path = evaluate.save_metrics(hybrid_metrics, 'hybrid_metrics.json')

hybrid_history_payload = {
    'model': 'GMM and CNN hybrid',
    'epochs_completed': int(epochs_completed),
    'best_epoch': int(best_epoch),
    'best_val_accuracy': float(best_val_accuracy),
    'training_seconds': float(hybrid_training_seconds),
    'segmentation_seconds': float(transform_seconds) if transform_seconds is not None else None,
    'parameters': int(hybrid_parameter_count),
    'batch_size': int(config.BATCH_SIZE),
    'steps_per_epoch': int(steps_per_epoch),
    'auc_scores': {key: float(value) for key, value in hybrid_auc_scores.items()},
    'clinical_error_summary': {key: float(value) for key, value in hybrid_clinical.items()},
    'mean_gradcam_mask_iou': float(agreement_scores.mean()),
    'history': {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    },
}
hybrid_history_path = evaluate.save_metrics(hybrid_history_payload, 'hybrid_history.json')

hybrid_model_path = config.MODELS_DIR / config.HYBRID_MODEL_NAME
hybrid_model.save(hybrid_model_path)

print('Metrics written to:', hybrid_metrics_path)
print('History written to:', hybrid_history_path)
print('Model written to:  ', hybrid_model_path)
print('Cached hybrid arrays:', cache_path)


## Summary

The hybrid pipeline has been trained and evaluated under conditions identical to those of the
baseline, with the Gaussian Mixture attention transformation as the sole point of difference. The
accuracy, weighted F1, macro F1, per class scores, clinically weighted error rates and bootstrap
intervals printed above, and stored in `results/metrics/hybrid_metrics.json`, are the figures that
answer the research question when placed beside their baseline counterparts.

The hypothesis stated at the start of this notebook predicted three observable consequences: a higher
weighted F1 than the baseline, convergence to the best validation accuracy in fewer epochs, and a
specific reduction in glioma and meningioma confusion visible in the normalised confusion matrix. The
cells above print all three quantities together with their baseline counterparts, so the hypothesis
should be judged by direct comparison of those printed values rather than by assertion here. The
Grad-CAM analysis provides the mechanistic component of that judgement: if the mean intersection over
union is substantial, the network demonstrably used the region the mixture model identified, and a
performance gain can reasonably be attributed to the designed mechanism; if the overlap is negligible,
then whatever difference exists must be explained by some other effect of the intensity attenuation,
such as generic regularisation.

Two caveats restrict the strength of any conclusion drawn at this point. First, a difference in a
point estimate is not evidence of a real effect until it is tested against the sampling variability of
a finite test set; the bootstrap intervals above are indicative only, and the formal paired test is
McNemar's test, carried out in notebook 05. Second, the hybrid pipeline is not free: the per image
mixture fit reported in the transformation cell is incurred again for every scan at inference time, so
even a statistically significant improvement has to be weighed against that cost. Notebook 05
addresses both points and selects the final model.
